# ROSP-7 diagnostic: system DIQ from the delivered PSF minus the atmospheric seeing

Diagnostic notebook for PTH Tier 1 metric [ROSP-7](https://rubinobs.atlassian.net/browse/ROSP-7),
system delivered image quality (sDIQ). It estimates the **system** (telescope + dome + camera) contribution
to the delivered image quality by subtracting, in quadrature, an **atmospheric seeing monitor** from the
delivered PSF FWHM of LSSTCam science visits, and looks at how that residual depends on wind, pointing,
active-optics state and the seeing itself.

**Data**

* Delivered PSF: `psf_sigma_median` from the ConsDB (`cdb_lsstcam.visit1` + `visit1_quicklook`, with
  `observation_reason` from `cdb_lsstcam.exposure`; schema: https://sdm-schemas.lsst.io/cdb_lsstcam.html), converted to a FWHM with
  $\mathrm{FWHM} = \sigma \times 2\sqrt{2\ln 2} \times 0.2''/\mathrm{pix}$.
* Atmosphere, two options (both monitors report seeing at **zenith and 500 nm**):
  1. **DIMM, outer-scale corrected** (von Kármán, Tokovinin 2002) for $L_0 = 30$ m, and used only when the
     outside wind speed is at least `dimm_min_wind_speed` (6 m/s), below which the DIMM is known to be unreliable:
     $$ r_0 = \frac{0.98 \cdot 500\,\mathrm{nm}}{\theta_\mathrm{DIMM}\,[\mathrm{rad}]}, \qquad
        \theta_{L_0} = \theta_\mathrm{DIMM}\sqrt{1 - 2.18\,(r_0/L_0)^{0.356}} $$
  2. **RINGSS** scintillation seeing (`fwhmScintillation` from `lsst.sal.ESS.logevent_ringssMeasurement`,
     `salIndex = 304`, in the EFD), the median of the measurements within `ringss_window_minutes` of the
     visit midpoint. If the EFD is unreachable the notebook falls back to the ConsDB `ringss_seeing` column.

**Method**

The monitor seeing is *de-corrected* from zenith / 500 nm to the airmass $X$ and band of each visit, the
inverse of the `lsst.summit.utils` zenith / 500 nm corrections, so the subtraction is done in the observed frame
where the system terms live:

$$ \theta_\mathrm{atm,obs} = \theta_\mathrm{atm}(\mathrm{zenith}, 500\,\mathrm{nm}) \times X^{0.6}
   \times \left(\lambda_\mathrm{eff}/500\,\mathrm{nm}\right)^{-0.2} $$

Per visit the **squared system term** is

$$ \mathrm{sys}^2 = \mathrm{FWHM}_\mathrm{obs}^2 - \theta_\mathrm{atm,obs}^2 \quad (\mathrm{arcsec}^2), $$

which is noisy and may be negative for individual visits. The sDIQ estimate for any set of visits (the whole
range, a night, or a bin of wind speed, ...) is the square root of the **median** of $\mathrm{sys}^2$ over
that set, taken *before* the square root so negative visits are kept and the noise averages down:

$$ \mathrm{sDIQ} = \sqrt{\operatorname{median}(\mathrm{sys}^2)}. $$

Where a median is negative the plots show the signed square root, $-\sqrt{|\cdot|}$, so the point stays
visible. Error bars are the robust standard error of the median of $\mathrm{sys}^2$
($1.2533 \times 1.4826\,\mathrm{MAD}/\sqrt{n}$) propagated through the square root.

**Outputs**: headline sDIQ per atmospheric metric, a histogram of $\mathrm{sys}^2$, the nightly sDIQ time
series, and sDIQ against wind speed, pointing relative to the wind, `aos_fwhm`, and the atmospheric seeing
metric itself.

**Parameters** (first code cell): the dayObs range (`start_dayobs` / `end_dayobs`, PTH convention), the
science programs, the DIMM outer scale and minimum wind, the RINGSS matching window, and the EFD name.

In [ ]:
# Parameters cell. Set defaults here - gets replaced by the sidecar on Times Square.
start_dayobs = 20251025  # First dayObs (inclusive)
end_dayobs = 20260715  # Last dayObs (inclusive)
science_programs = "407, 408, 416, 417, 419, 421"  # bare numbers become BLOCK-<n>
dimm_outer_scale_m = 30  # outer scale L0 (m) for the von Karman DIMM correction
dimm_min_wind_speed = (
    6.0  # DIMM-based sDIQ only uses visits with wind_speed >= this (m/s)
)
ringss_window_minutes = (
    5  # RINGSS measurements within +/- this of the visit midpoint are medianed
)
sdiq_goal_arcsec = 0.45  # reference line: nominal system-contribution goal used in image_quality_trending
efd_name = "usdf_efd"  # lsst_efd_client instance name
use_cache = True  # cache the ConsDB/EFD pulls as parquet under ./cache (ignored on Times Square)

not_times_square = True

In [ ]:
# Times Square runs this notebook in a Nublado pod where rubin_nights may be
# missing or stale; install/upgrade it there. Anywhere else this cell is a no-op.
try:
    not_times_square
except NameError:
    not_times_square = False

if not not_times_square:
    print("updating rubin_nights")
    !pip install --user --upgrade git+https://github.com/lsst-sims/rubin_nights.git --no-deps > /dev/null 2>&1

In [ ]:
import inspect
import os
import re
from dataclasses import dataclass
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from astropy.time import Time

%matplotlib inline
pd.set_option("display.width", 160)
NOTEBOOK_DIR = os.getcwd()

## Constants and parameter helpers

In [ ]:
DEFAULT_PROGRAMS = (
    "BLOCK-407",
    "BLOCK-408",
    "BLOCK-416",
    "BLOCK-417",
    "BLOCK-419",
    "BLOCK-421",
)

# Bump when the ConsDB query changes so parquet caches written with the old column set are not reused.
CACHE_VERSION = "v2"

SIGMA_TO_FWHM = 2.0 * np.sqrt(2.0 * np.log(2.0))
PLATESCALE = 0.2  # arcsec / pixel, LSSTCam
ARCSEC_TO_RAD = np.pi / (180.0 * 3600.0)
REFERENCE_WAVELENGTH_NM = 500.0
CAM_FWHM = 0.207  # arcsec, camera diffusion term used in image_quality_trending (reference line only)

# Effective wavelengths used by lsst.summit.utils.getBandpassSeeingCorrection for LSSTCam.
LSSTCAM_EFFECTIVE_WAVELENGTH_NM = {
    "u": 365.1397,
    "g": 475.6104,
    "r": 618.6055,
    "i": 753.5490,
    "z": 868.0935,
    "y": 972.5979,
}
BAND_ORDER = ("u", "g", "r", "i", "z", "y")

# Columns pulled from cdb_lsstcam.visit1 (per-visit metadata) and visit1_quicklook (image quality).
VISIT_COLUMNS = (
    "visit_id",
    "day_obs",
    "seq_num",
    "band",
    "physical_filter",
    "science_program",
    "img_type",
    "target_name",
    "exp_time",
    "exp_midpt",
    "exp_midpt_mjd",
    "airmass",
    "altitude",
    "azimuth",
    "zenith_distance",
    "air_temp",
    "pressure",
    "humidity",
    "wind_speed",
    "wind_dir",
    "dimm_seeing",
)
QUICKLOOK_COLUMNS = (
    "psf_sigma_median",
    "psf_sigma_min",
    "psf_sigma_max",
    "n_psf_star_total",
    "aos_fwhm",
    "donut_blur_fwhm",
    "ringss_seeing",
)

# Columns pulled from cdb_lsstcam.exposure (joined on exposure_id = visit_id, the 1-to-1 visit system).
EXPOSURE_COLUMNS = ("observation_reason",)

# RINGSS event in the EFD.
RINGSS_TOPIC = "lsst.sal.ESS.logevent_ringssMeasurement"
RINGSS_SAL_INDEX = 304
RINGSS_FIELDS = (
    "timestamp",
    "hrNum",
    "zenithDistance",
    "flux",
    "fwhmScintillation",
    "fwhmSector",
    "fwhmFree",
    "wind",
)

# Chart palette: fixed categorical order (blue, orange, aqua, yellow, magenta, green, violet, red).
SERIES_COLORS = (
    "#2a78d6",
    "#eb6834",
    "#1baf7a",
    "#eda100",
    "#e87ba4",
    "#008300",
    "#4a3aa7",
    "#e34948",
)
SERIES_MARKERS = ("o", "s", "^", "D", "v", "P", "X", "*")
INK, INK_SECONDARY, INK_MUTED, GRID, SURFACE = (
    "#0b0b0b",
    "#52514e",
    "#898781",
    "#e1e0d9",
    "#fcfcfb",
)
GOAL_COLOR = "#1baf7a"
SEQUENTIAL_BLUE = LinearSegmentedColormap.from_list(
    "seq_blue",
    [
        "#ffffff",
        "#cde2fb",
        "#9ec5f4",
        "#6da7ec",
        "#3987e5",
        "#256abf",
        "#1c5cab",
        "#104281",
        "#0d366b",
    ],
)

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#c3c2b7",
        "axes.labelcolor": INK,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "xtick.color": INK_SECONDARY,
        "ytick.color": INK_SECONDARY,
        "legend.frameon": False,
        "font.size": 11,
        "axes.titlesize": 12,
        "lines.linewidth": 2.0,
        "savefig.dpi": 150,
    }
)


def parse_day_obs(value) -> int:
    """Coerce 20260101, "20260101" or "2026-01-01" to an int."""
    s = str(value).strip().replace("-", "")
    if not re.fullmatch(r"\d{8}", s):
        raise ValueError(f"Not a day_obs: {value!r}")
    return int(s)


def parse_programs(value) -> list:
    """List or comma/space separated string of block names; bare numbers become BLOCK-<n>."""
    if value is None:
        return list(DEFAULT_PROGRAMS)
    items = (
        [v for v in re.split(r"[,\s]+", value.strip()) if v]
        if isinstance(value, str)
        else [str(v).strip() for v in value]
    )
    return [f"BLOCK-{it}" if re.fullmatch(r"T?\d+", it) else it for it in items]


def parse_bool(value) -> bool:
    return str(value).strip().lower() in ("true", "1", "yes")


def day_obs_chunks(day_obs_min: int, day_obs_max: int, chunk_days: int):
    """Yield (lo, hi) day_obs pairs covering the range in calendar chunks."""
    lo, end = pd.Timestamp(str(day_obs_min)), pd.Timestamp(str(day_obs_max))
    while lo <= end:
        hi = min(lo + pd.Timedelta(days=chunk_days - 1), end)
        yield int(lo.strftime("%Y%m%d")), int(hi.strftime("%Y%m%d"))
        lo = hi + pd.Timedelta(days=1)


def day_obs_window(day_obs_lo: int, day_obs_hi: int):
    """UTC start/end (astropy Time) covering the nights day_obs_lo..day_obs_hi: 12:00 UTC to 12:00 UTC next day."""
    t0 = datetime.strptime(str(day_obs_lo), "%Y%m%d").replace(
        hour=12, tzinfo=timezone.utc
    )
    t1 = datetime.strptime(str(day_obs_hi), "%Y%m%d").replace(
        hour=12, tzinfo=timezone.utc
    ) + pd.Timedelta(days=1)
    return Time(t0), Time(t1)


def _read_cache(cache_path, use_cache):
    if cache_path and use_cache and os.path.exists(cache_path):
        df = pd.read_parquet(cache_path)
        print(f"Loaded {len(df)} rows from cache {cache_path}")
        return df
    return None


def _write_cache(cache_path, df):
    if cache_path and len(df):
        os.makedirs(os.path.dirname(os.path.abspath(cache_path)), exist_ok=True)
        df.to_parquet(cache_path, index=False)
        print(f"Saved {len(df)} rows to {cache_path}")
    return df


def cached(cache_path, loader, use_cache=True) -> pd.DataFrame:
    """Run loader() unless cache_path already holds its parquet output."""
    df = _read_cache(cache_path, use_cache)
    return df if df is not None else _write_cache(cache_path, loader())


async def cached_async(cache_path, loader, use_cache=True) -> pd.DataFrame:
    """Same as cached() for a loader that returns a coroutine (the EFD client is async on the RSP)."""
    df = _read_cache(cache_path, use_cache)
    return df if df is not None else _write_cache(cache_path, await loader())

## ConsDB access

One row per science visit (`visit1` + `visit1_quicklook`, plus `exposure` for `observation_reason`, joined on
`exposure_id = visit_id`). The client comes from `rubin_nights` (which resolves the endpoint and token for
Times Square, the USDF RSP or an SDF login node), falling back to `lsst.summit.utils.ConsDbClient`; the
query is chunked by calendar month to stay under the API timeout.

In [ ]:
USDF_CONSDB_URL = "http://consdb-pq.consdb:8080/consdb"


class _SummitUtilsClient:
    """Adapter so lsst.summit.utils.ConsDbClient.query returns a DataFrame."""

    def __init__(self, url=USDF_CONSDB_URL):
        from lsst.summit.utils import ConsDbClient

        os.environ["no_proxy"] = os.environ.get("no_proxy", "") + ",.consdb"
        self._client = ConsDbClient(url)

    def query(self, sql):
        result = self._client.query(sql)
        return result if isinstance(result, pd.DataFrame) else result.to_pandas()


def get_consdb_client():
    """Object with .query(sql) -> DataFrame: rubin_nights first, then summit_utils."""
    errors = []
    try:
        from rubin_nights import connections

        client = connections.get_clients()["consdb"]
        print("ConsDB client: rubin_nights")
        return client
    except Exception as exc:
        errors.append(f"rubin_nights: {exc!r}")
    try:
        client = _SummitUtilsClient()
        print("ConsDB client: lsst.summit.utils")
        return client
    except Exception as exc:
        errors.append(f"summit_utils: {exc!r}")
    raise RuntimeError("Could not build a ConsDB client:\n  " + "\n  ".join(errors))


def _sql_list(values):
    return ", ".join("'" + str(v).replace("'", "''") + "'" for v in values)


def build_query(
    day_obs_min,
    day_obs_max,
    programs=DEFAULT_PROGRAMS,
    instrument="lsstcam",
    img_types=("science",),
):
    """SQL joining visit1 (pointing, weather, DIMM), visit1_quicklook (PSF stats) and exposure (observation_reason), one row per visit."""
    schema = f"cdb_{instrument}"
    where = [f"v.day_obs >= {int(day_obs_min)}", f"v.day_obs <= {int(day_obs_max)}"]
    if programs:
        where.append(f"v.science_program IN ({_sql_list(programs)})")
    if img_types:
        where.append(f"v.img_type IN ({_sql_list(img_types)})")
    return (
        "SELECT\n    "
        + ",\n    ".join(
            [f"v.{c}" for c in VISIT_COLUMNS]
            + [f"q.{c}" for c in QUICKLOOK_COLUMNS]
            + [f"e.{c}" for c in EXPOSURE_COLUMNS]
        )
        + f"\nFROM {schema}.visit1 AS v"
        + f"\nLEFT JOIN {schema}.visit1_quicklook AS q ON q.visit_id = v.visit_id"
        + f"\nLEFT JOIN {schema}.exposure AS e ON e.exposure_id = v.visit_id"
        + "\nWHERE "
        + "\n  AND ".join(where)
        + "\nORDER BY v.day_obs, v.seq_num"
    )


def fetch_visits(
    client, day_obs_min, day_obs_max, programs=DEFAULT_PROGRAMS, chunk_days=31
):
    frames = []
    for lo, hi in day_obs_chunks(day_obs_min, day_obs_max, chunk_days):
        df = client.query(build_query(lo, hi, programs))
        print(f"  consdb {lo}-{hi}: {len(df):6d} visits")
        if len(df):
            frames.append(df)
    if not frames:
        return pd.DataFrame(
            columns=list(VISIT_COLUMNS)
            + list(QUICKLOOK_COLUMNS)
            + list(EXPOSURE_COLUMNS)
        )
    out = pd.concat(frames, ignore_index=True).sort_values(["day_obs", "seq_num"])
    missing = [
        c for c in VISIT_COLUMNS + QUICKLOOK_COLUMNS + EXPOSURE_COLUMNS if c not in out
    ]
    if missing:
        print(f"WARNING: columns not returned by the ConsDB: {missing}")
    for c in out.columns:
        if (
            c
            not in (
                "band",
                "physical_filter",
                "science_program",
                "observation_reason",
                "img_type",
                "target_name",
                "exp_midpt",
            )
            and out[c].dtype == object
        ):
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out.reset_index(drop=True)

## EFD access: RINGSS

`fetch_ringss` pulls every `logevent_ringssMeasurement` (salIndex 304) over the night range with
`lsst_efd_client`. `match_ringss_to_visits` attaches to each visit the median of the measurements within
± `ringss_window_minutes` of the exposure midpoint (`ringss_scint`, `ringss_sector`, `ringss_free`,
`ringss_wind`, `ringss_zd`) and the number used (`ringss_n`). `load_ringss` wraps the EFD pull in a
fallback: if the EFD cannot be reached or returns nothing, the ConsDB `ringss_seeing` column is used instead
(flagged by `ringss_source`).

In [ ]:
def get_efd_client(efd_name="usdf_efd"):
    import lsst_efd_client

    return lsst_efd_client.EfdClient(efd_name, output_mode="dataframe")


async def fetch_ringss(
    efd_client, day_obs_min, day_obs_max, chunk_days=31, sal_index=RINGSS_SAL_INDEX
):
    """All RINGSS measurements between 12:00 UTC on day_obs_min and 12:00 UTC the day after day_obs_max."""
    frames = []
    for lo, hi in day_obs_chunks(day_obs_min, day_obs_max, chunk_days):
        start, end = day_obs_window(lo, hi)
        df = efd_client.select_time_series(
            RINGSS_TOPIC, list(RINGSS_FIELDS), start, end, index=sal_index
        )
        if inspect.isawaitable(df):
            df = await df
        print(f"  ringss {lo}-{hi}: {len(df):6d} measurements")
        if len(df):
            df = df.reset_index().rename(columns={df.index.name or "index": "time"})
            frames.append(df)
    if not frames:
        return pd.DataFrame(columns=["time", *RINGSS_FIELDS])
    out = pd.concat(frames, ignore_index=True)
    out["time"] = pd.to_datetime(out["time"], utc=True).dt.tz_localize(None)
    return out.sort_values("time").reset_index(drop=True)


async def load_ringss(
    efd_name, day_obs_min, day_obs_max, cache_path=None, use_cache=True
):
    """RINGSS table from the EFD, or None (with a warning) if the EFD is unavailable."""
    try:
        efd = get_efd_client(efd_name)
        ringss = await cached_async(
            cache_path, lambda: fetch_ringss(efd, day_obs_min, day_obs_max), use_cache
        )
    except Exception as exc:
        print(
            f"WARNING: RINGSS pull from the EFD failed ({exc!r}); "
            "falling back to the ConsDB ringss_seeing column."
        )
        return None
    if ringss is None or len(ringss) == 0:
        print(
            "WARNING: the EFD returned no RINGSS measurements; "
            "falling back to the ConsDB ringss_seeing column."
        )
        return None
    return ringss


def match_ringss_to_visits(visits, ringss, window_minutes=5.0):
    """Median RINGSS quantities within +/- window of each visit midpoint.

    Returns a copy of ``visits`` with ringss_scint, ringss_sector, ringss_free, ringss_wind,
    ringss_zd, ringss_n and ringss_source ("efd" or "consdb"). With ``ringss=None`` the ConsDB
    ``ringss_seeing`` column stands in for ringss_scint.
    """
    out = visits.copy()
    cols = {
        "ringss_scint": "fwhmScintillation",
        "ringss_sector": "fwhmSector",
        "ringss_free": "fwhmFree",
        "ringss_wind": "wind",
        "ringss_zd": "zenithDistance",
    }
    for c in cols:
        out[c] = np.nan
    out["ringss_n"] = 0
    if ringss is None or len(ringss) == 0:
        out["ringss_scint"] = pd.to_numeric(out["ringss_seeing"], errors="coerce")
        out["ringss_n"] = out["ringss_scint"].notna().astype(int)
        out["ringss_source"] = "consdb"
        return out
    out["ringss_source"] = "efd"
    r = ringss.dropna(subset=["time"]).sort_values("time")
    t_r = r["time"].to_numpy(dtype="datetime64[ns]").astype("int64") / 1e9
    vals = {
        c: r[src].to_numpy(dtype=float) if src in r else np.full(len(r), np.nan)
        for c, src in cols.items()
    }
    t_v = (
        pd.to_datetime(out["exp_midpt"], utc=True, format="ISO8601")
        .dt.tz_localize(None)
        .to_numpy(dtype="datetime64[ns]")
        .astype("int64")
        / 1e9
    )
    w = window_minutes * 60.0
    lo = np.searchsorted(t_r, t_v - w, side="left")
    hi = np.searchsorted(t_r, t_v + w, side="right")
    res = {c: np.full(len(out), np.nan) for c in cols}
    n = np.zeros(len(out), dtype=int)
    for i, (a, b) in enumerate(zip(lo, hi)):
        if b > a:
            n[i] = b - a
            for c in cols:
                seg = vals[c][a:b]
                seg = seg[np.isfinite(seg)]
                if len(seg):
                    res[c][i] = np.median(seg)
    for c in cols:
        out[c] = res[c]
    out["ringss_n"] = n
    return out

## Physics: outer-scale DIMM correction, de-correction to the observed frame, squared system term

An *atmospheric metric* is one monitor-based seeing estimate. Each has a tag, a label and three columns:
`<tag>_z500` (the monitor value at zenith / 500 nm), `atm_<tag>` (de-corrected to the airmass and band of
the visit) and `sys2_<tag>` = $\mathrm{FWHM}_\mathrm{obs}^2 - \mathrm{atm}_\mathrm{<tag>}^2$ in arcsec². The DIMM
metric is blanked (NaN) on visits below `dimm_min_wind_speed`.

In [ ]:
def airmass_seeing_correction(airmass):
    """Multiply an observed FWHM by this to refer it to zenith (X^-0.6)."""
    airmass = np.asarray(airmass, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(airmass >= 1.0, airmass ** (-0.6), np.nan)


def bandpass_seeing_correction(band):
    """Multiply an observed FWHM by this to refer it to 500 nm; accepts 'r' or 'r_57'."""
    first = pd.Series(band).astype("string").str.strip().str[0]
    wl = first.map(LSSTCAM_EFFECTIVE_WAVELENGTH_NM).astype(float)
    return (wl.to_numpy() / REFERENCE_WAVELENGTH_NM) ** 0.2


def seeing_zenith_500nm_to_observed(seeing_z500, airmass, band):
    """De-correct a zenith / 500 nm seeing to the airmass and band of a visit: X^0.6 (lambda/500nm)^-0.2."""
    seeing = np.asarray(seeing_z500, dtype=float)
    return (
        seeing / airmass_seeing_correction(airmass) / bandpass_seeing_correction(band)
    )


def fried_parameter(seeing_arcsec, wavelength_nm=REFERENCE_WAVELENGTH_NM):
    """r0 (m) from a Kolmogorov FWHM in arcsec: r0 = 0.98 lambda / FWHM."""
    with np.errstate(divide="ignore", invalid="ignore"):
        return (
            0.98
            * wavelength_nm
            * 1e-9
            / (np.asarray(seeing_arcsec, dtype=float) * ARCSEC_TO_RAD)
        )


def outer_scale_correction(seeing_arcsec, L0):
    """sqrt(1 - 2.18 (r0/L0)^0.356) (Tokovinin 2002); 1 for L0=None; NaN where the bracket is negative."""
    seeing = np.asarray(seeing_arcsec, dtype=float)
    if L0 is None:
        return np.where(np.isfinite(seeing), 1.0, np.nan)
    with np.errstate(invalid="ignore"):
        bracket = 1.0 - 2.18 * np.power(fried_parameter(seeing) / float(L0), 0.356)
        return np.where(bracket > 0, np.sqrt(np.clip(bracket, 0, None)), np.nan)


def signed_sqrt(x):
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.sqrt(np.abs(x))


@dataclass
class AtmMetric:
    tag: str
    label: str
    short: str
    color: str
    marker: str

    @property
    def z500(self):
        return f"{self.tag}_z500"

    @property
    def atm(self):
        return f"atm_{self.tag}"

    @property
    def sys2(self):
        return f"sys2_{self.tag}"


def make_metrics(L0, dimm_min_wind):
    return [
        AtmMetric(
            "dimm",
            f"DIMM, L0 = {L0:g} m (wind >= {dimm_min_wind:g} m/s)",
            "DIMM L0",
            SERIES_COLORS[0],
            SERIES_MARKERS[0],
        ),
        AtmMetric(
            "ringss",
            "RINGSS scintillation",
            "RINGSS",
            SERIES_COLORS[1],
            SERIES_MARKERS[1],
        ),
    ]


def add_derived_columns(df, metrics, L0, dimm_min_wind):
    """Observed FWHM, each metric at zenith/500nm and in the observed frame, and sys2 per metric."""
    out = df.copy()
    out["psf_fwhm"] = out["psf_sigma_median"].astype(float) * SIGMA_TO_FWHM * PLATESCALE
    out["fwhm_zenith_500nm"] = (
        out["psf_fwhm"]
        * airmass_seeing_correction(out["airmass"])
        * bandpass_seeing_correction(out["band"])
    )
    out["dimm_r0"] = fried_parameter(out["dimm_seeing"])
    out["dimm_z500"] = out["dimm_seeing"].to_numpy(
        dtype=float
    ) * outer_scale_correction(out["dimm_seeing"], L0)
    out["dimm_windy"] = out["wind_speed"] >= dimm_min_wind
    out.loc[~out["dimm_windy"].fillna(False), "dimm_z500"] = np.nan
    out["ringss_z500"] = out["ringss_scint"].astype(float)
    for m in metrics:
        out[m.atm] = seeing_zenith_500nm_to_observed(
            out[m.z500], out["airmass"], out["band"]
        )
        out[m.sys2] = out["psf_fwhm"] ** 2 - out[m.atm] ** 2
    out["time"] = pd.to_datetime(
        out["exp_midpt"], utc=True, errors="coerce"
    ).dt.tz_localize(None)
    out["date"] = pd.to_datetime(out["day_obs"].astype(int).astype(str))
    out["rel_wind_az"] = (
        (out["azimuth"] - out["wind_dir"] + 180.0) % 360.0
    ) - 180.0  # 0 = into the wind
    out["abs_rel_wind_az"] = out["rel_wind_az"].abs()
    return out


@dataclass
class QualityCuts:
    fwhm_range: tuple = (0.4, 4.0)
    dimm_range: tuple = (0.3, 4.0)
    ringss_range: tuple = (0.2, 4.0)
    airmass_max: float = 2.5
    exclude_bands: tuple = ()
    exclude_reasons: tuple = (
        "filter_change_close_loop",
    )  # substring match on observation_reason
    require_both: bool = (
        False  # True: keep only visits where both metrics are valid (common sample)
    )


def apply_quality_cuts(df, metrics, cuts=QualityCuts(), verbose=True):
    """Drop visits without a usable PSF/wind; blank (not drop) each metric where its monitor is out of range."""
    steps = [
        ("has PSF FWHM", df["psf_fwhm"].notna()),
        (f"PSF FWHM in {cuts.fwhm_range}", df["psf_fwhm"].between(*cuts.fwhm_range)),
        (f"airmass <= {cuts.airmass_max}", df["airmass"] <= cuts.airmass_max),
        ("has wind speed", df["wind_speed"].notna()),
    ]
    if cuts.exclude_bands:
        steps.append(
            (f"band not in {cuts.exclude_bands}", ~df["band"].isin(cuts.exclude_bands))
        )
    if "observation_reason" in df:
        for reason in cuts.exclude_reasons:
            steps.append(
                (
                    f"observation_reason without '{reason}'",
                    ~df["observation_reason"]
                    .astype("string")
                    .fillna("")
                    .str.contains(reason),
                )
            )
    elif cuts.exclude_reasons:
        print(
            "NOTE: no observation_reason column (stale cache or query change); "
            f"the {cuts.exclude_reasons} cut is skipped."
        )
    dimm_ok = df["dimm_seeing"].between(*cuts.dimm_range) & df["dimm_z500"].notna()
    ringss_ok = df["ringss_scint"].between(*cuts.ringss_range)
    if cuts.require_both:
        steps.append(("DIMM and RINGSS both valid", dimm_ok & ringss_ok))
    else:
        steps.append(("DIMM or RINGSS valid", dimm_ok | ringss_ok))
    keep = pd.Series(True, index=df.index)
    rows = []
    for name, mask in steps:
        mask = mask.fillna(False).astype(bool)
        rows.append((name, int((keep & ~mask).sum()), int((keep & mask).sum())))
        keep &= mask
    out = df[keep].copy()
    by_tag = {"dimm": dimm_ok, "ringss": ringss_ok}
    for m in metrics:
        bad = ~by_tag[m.tag][keep].fillna(False).astype(bool)
        out.loc[bad, [m.z500, m.atm, m.sys2]] = np.nan
    if verbose:
        print(f"Start: {len(df)} visits")
        print(
            pd.DataFrame(rows, columns=["cut", "removed", "remaining"]).to_string(
                index=False
            )
        )
        print("\nvisits with a valid metric:")
        for m in metrics:
            print(f"  {m.label}: {int(out[m.sys2].notna().sum())}")
    return out

## Statistics and plotting helpers

`sdiq_stats` is the core estimator: the median of $\mathrm{sys}^2$ over a set of visits, its robust standard
error, and the (signed) square root. `binned_sdiq` applies it in bins of any x variable; `nightly_sdiq` per
night.

In [ ]:
def sdiq_stats(sys2, min_count=1):
    """Median of sys2 (arcsec^2), robust SE of the median, and the signed-sqrt sDIQ with propagated error."""
    s = np.asarray(sys2, dtype=float)
    s = s[np.isfinite(s)]
    row = dict(
        n=len(s),
        median_sys2=np.nan,
        median_sys2_err=np.nan,
        p16_sys2=np.nan,
        p84_sys2=np.nan,
        frac_negative=np.nan,
        sdiq=np.nan,
        sdiq_err=np.nan,
    )
    if len(s) < min_count or len(s) == 0:
        return row
    med = np.median(s)
    mad = np.median(np.abs(s - med))
    err = 1.2533 * 1.4826 * mad / np.sqrt(len(s))
    row.update(
        median_sys2=med,
        median_sys2_err=err,
        p16_sys2=np.percentile(s, 16),
        p84_sys2=np.percentile(s, 84),
        frac_negative=float((s < 0).mean()),
        sdiq=float(signed_sqrt(med)),
        sdiq_err=err / (2.0 * np.sqrt(abs(med))) if med != 0 else np.nan,
    )
    return row


def binned_sdiq(x, sys2, bins, min_count=10):
    """sdiq_stats of sys2 in bins of x; rows with n < min_count are left NaN."""
    x, s = np.asarray(x, dtype=float), np.asarray(sys2, dtype=float)
    ok = np.isfinite(x) & np.isfinite(s)
    x, s = x[ok], s[ok]
    edges = np.asarray(bins, dtype=float)
    idx = np.digitize(x, edges) - 1
    rows = []
    for i in range(len(edges) - 1):
        sel = s[idx == i]
        row = dict(lo=edges[i], hi=edges[i + 1], center=0.5 * (edges[i] + edges[i + 1]))
        row.update(sdiq_stats(sel, min_count=min_count))
        rows.append(row)
    return pd.DataFrame(rows)


def summary_table(df, metrics):
    rows = []
    for m in metrics:
        ok = df[m.sys2].notna()
        st = sdiq_stats(df.loc[ok, m.sys2])
        rows.append(
            dict(
                metric=m.label,
                n_visits=st["n"],
                n_nights=int(df.loc[ok, "day_obs"].nunique()),
                median_psf_fwhm=df.loc[ok, "psf_fwhm"].median(),
                median_atm_obs=df.loc[ok, m.atm].median(),
                median_atm_z500=df.loc[ok, m.z500].median(),
                median_sys2=st["median_sys2"],
                median_sys2_err=st["median_sys2_err"],
                p16_sys2=st["p16_sys2"],
                p84_sys2=st["p84_sys2"],
                frac_sys2_negative=st["frac_negative"],
                sdiq=st["sdiq"],
                sdiq_err=st["sdiq_err"],
            )
        )
    return pd.DataFrame(rows).set_index("metric")


def summary_by_group(df, metrics, group_col, min_count=10):
    """sDIQ per metric for each value of group_col (band, science_program, ...)."""
    rows = []
    for key, sub in df.groupby(group_col):
        row = {group_col: key, "n_visits": len(sub)}
        for m in metrics:
            st = sdiq_stats(sub[m.sys2], min_count=min_count)
            row[f"n_{m.tag}"] = st["n"]
            row[f"sdiq_{m.tag}"] = st["sdiq"]
            row[f"sdiq_err_{m.tag}"] = st["sdiq_err"]
        rows.append(row)
    out = pd.DataFrame(rows).set_index(group_col)
    if group_col == "band":
        out = out.reindex([b for b in BAND_ORDER if b in out.index])
    return out


def nightly_sdiq(df, metrics, min_count=5):
    """One row per night: visit counts, median wind, median FWHM, each metric's median seeing and sDIQ."""
    rows = []
    for day_obs, sub in df.groupby("day_obs"):
        row = dict(
            day_obs=int(day_obs),
            date=pd.Timestamp(str(int(day_obs))),
            n_visits=len(sub),
            wind_speed=sub["wind_speed"].median(),
            psf_fwhm=sub["psf_fwhm"].median(),
            aos_fwhm=sub["aos_fwhm"].median(),
        )
        for m in metrics:
            st = sdiq_stats(sub[m.sys2], min_count=min_count)
            row[m.atm] = sub[m.atm].median()
            row[f"n_{m.tag}"] = st["n"]
            row[f"median_sys2_{m.tag}"] = st["median_sys2"]
            row[f"sdiq_{m.tag}"] = st["sdiq"]
            row[f"sdiq_err_{m.tag}"] = st["sdiq_err"]
        rows.append(row)
    return pd.DataFrame(rows)


def _break_gaps(nightly, max_gap_days=3):
    """Insert NaN rows after gaps longer than max_gap_days so lines break instead of bridging."""
    nightly = nightly.sort_values("date").reset_index(drop=True)
    gaps = nightly["date"].diff().dt.days > max_gap_days
    if not gaps.any():
        return nightly
    pieces = []
    for i, (_, row) in enumerate(nightly.iterrows()):
        if gaps.iloc[i]:
            blank = row.copy()
            blank[:] = np.nan
            blank["date"] = row["date"] - pd.Timedelta(days=1)
            pieces.append(blank)
        pieces.append(row)
    return pd.DataFrame(pieces).reset_index(drop=True)


def plot_sys2_histogram(df, metrics, bins=np.arange(-1.0, 2.0, 0.025), goal=None):
    """Histogram of the per-visit squared system term, one curve per metric, with the medians marked."""
    fig, ax = plt.subplots(figsize=(9, 5))
    for m in metrics:
        s = df[m.sys2].dropna()
        if not len(s):
            continue
        st = sdiq_stats(s)
        ax.hist(
            s.clip(bins[0], bins[-1]),
            bins=bins,
            histtype="step",
            lw=2,
            color=m.color,
            label=f'{m.label}: median {st["median_sys2"]:+.3f} arcsec² -> sDIQ {st["sdiq"]:.3f}" (n = {st["n"]})',
        )
        ax.axvline(st["median_sys2"], color=m.color, lw=1.2, ls=":")
    ax.axvline(0, color=INK_MUTED, lw=1, ls="--")
    if goal:
        ax.axvline(
            goal**2,
            color=GOAL_COLOR,
            lw=1.5,
            ls="--",
            label=f'goal {goal:g}" (= {goal**2:.3f} arcsec²)',
        )
    ax.set_xlabel("PSF FWHM² - atmosphere² (arcsec²)")
    ax.set_ylabel("visits")
    ax.set_xlim(bins[0], bins[-1])
    ax.legend(loc="upper left", fontsize=9)
    return fig


def plot_nightly(df, metrics, goal=None, min_count=5):
    """Nightly medians: FWHM and atmosphere (top), sDIQ per metric (middle), visits and wind (bottom)."""
    nightly = _break_gaps(nightly_sdiq(df, metrics, min_count=min_count))
    fig, axes = plt.subplots(
        3,
        1,
        figsize=(13, 10),
        sharex=True,
        gridspec_kw={"height_ratios": [2, 2.4, 1.2]},
    )
    ax = axes[0]
    ax.plot(
        nightly["date"],
        nightly["psf_fwhm"],
        marker="o",
        ms=4,
        lw=1.2,
        color=INK,
        label="delivered PSF FWHM",
    )
    for m in metrics:
        ax.plot(
            nightly["date"],
            nightly[m.atm],
            marker=m.marker,
            ms=4,
            lw=1.2,
            color=m.color,
            label=f"{m.short} (de-corrected to visit airmass/band)",
        )
    ax.set_ylabel("nightly median (arcsec)")
    ax.legend(ncol=3, fontsize=9)
    ax = axes[1]
    for m in metrics:
        ax.errorbar(
            nightly["date"],
            nightly[f"sdiq_{m.tag}"],
            yerr=nightly[f"sdiq_err_{m.tag}"],
            marker=m.marker,
            ms=5,
            lw=1.2,
            capsize=2,
            color=m.color,
            label=m.label,
        )
    ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
    if goal:
        ax.axhline(goal, color=GOAL_COLOR, lw=1.5, ls="--", label=f'goal {goal:g}"')
        ax.axhspan(0, goal, color=GOAL_COLOR, alpha=0.08, lw=0)
    ax.set_ylabel("sDIQ = sqrt(median FWHM² - atm²) (arcsec)")
    ax.legend(ncol=3, fontsize=9)
    ax = axes[2]
    ax.bar(nightly["date"], nightly["n_visits"], width=0.8, color=GRID, label="visits")
    ax.set_ylabel("visits")
    ax2 = ax.twinx()
    ax2.plot(
        nightly["date"],
        nightly["wind_speed"],
        marker="o",
        ms=3,
        lw=1.2,
        color=SERIES_COLORS[2],
    )
    ax2.set_ylabel("median wind (m/s)", color=SERIES_COLORS[2])
    ax2.grid(False)
    ax.set_xlabel("night (day_obs)")
    fig.align_ylabels(axes)
    return fig, nightly


def plot_sdiq_vs(
    df,
    metrics,
    xcol,
    bins,
    xlabel,
    ylim_sys2=(-0.8, 1.6),
    ylim_sdiq=None,
    gridsize=40,
    min_count=10,
    goal=None,
    ref_lines=(),
    title=None,
    mask=None,
):
    """sys2 vs x: hexbin per metric with the binned median (top), sDIQ vs x for all metrics (middle), counts (bottom).

    ``xcol`` may be a column name or a dict metric.tag -> column name (x differs per metric).
    ``ref_lines`` is a list of (xgrid, ygrid, label) reference curves for the sDIQ panel.
    """
    n = len(metrics)
    edges = np.asarray(bins, dtype=float)
    xmin, xmax = float(edges[0]), float(edges[-1])
    fig = plt.figure(figsize=(5.4 * n, 11.5), layout="constrained")
    gs = fig.add_gridspec(3, n, height_ratios=[3.2, 3.2, 1.1])
    hex_axes = [fig.add_subplot(gs[0, k]) for k in range(n)]
    ax_sdiq = fig.add_subplot(gs[1, :])
    ax_cnt = fig.add_subplot(gs[2, :], sharex=ax_sdiq)
    hb = None
    tables = {}
    for ax, m in zip(hex_axes, metrics):
        xc = xcol[m.tag] if isinstance(xcol, dict) else xcol
        ok = df[m.sys2].notna() & df[xc].notna()
        if mask is not None:
            ok &= mask
        x, y = df.loc[ok, xc], df.loc[ok, m.sys2]
        if ok.sum():
            hb = ax.hexbin(
                x,
                y,
                gridsize=gridsize,
                extent=(xmin, xmax, *ylim_sys2),
                cmap=SEQUENTIAL_BLUE,
                norm=LogNorm(vmin=1),
                mincnt=1,
                linewidths=0.2,
                edgecolors="face",
            )
        st = binned_sdiq(x, y, edges, min_count=min_count)
        tables[m.tag] = st
        ax.fill_between(
            st["center"],
            st["p16_sys2"],
            st["p84_sys2"],
            color=m.color,
            alpha=0.15,
            lw=0,
        )
        ax.plot(
            st["center"], st["median_sys2"], color=SURFACE, lw=4, solid_capstyle="round"
        )
        ax.plot(
            st["center"],
            st["median_sys2"],
            color=m.color,
            lw=2,
            marker="o",
            ms=4,
            label="binned median (16-84 % band)",
        )
        ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
        ax.set_title(f"{m.label}   (n = {int(ok.sum())})", fontsize=11)
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(*ylim_sys2)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("FWHM² - atm² (arcsec²)")
    hex_axes[0].legend(loc="upper left", fontsize=9)
    if hb is not None:
        cbar = fig.colorbar(hb, ax=hex_axes, pad=0.02, shrink=0.9)
        cbar.set_label("visits per hexagon")
    for m in metrics:
        st = tables[m.tag]
        ax_sdiq.errorbar(
            st["center"],
            st["sdiq"],
            yerr=st["sdiq_err"],
            color=m.color,
            marker=m.marker,
            ms=6,
            capsize=2,
            lw=2,
            label=m.label,
        )
    for k, (xg, yg, lab) in enumerate(ref_lines):
        ax_sdiq.plot(
            xg, yg, color=INK_MUTED, lw=1.4, ls=(":", "-.", "--")[k % 3], label=lab
        )
    ax_sdiq.axhline(0, color=INK_MUTED, lw=1, ls="--")
    if goal:
        ax_sdiq.axhline(
            goal, color=GOAL_COLOR, lw=1.5, ls="--", label=f'goal {goal:g}"'
        )
    ax_sdiq.set_ylabel("sDIQ = sqrt(binned median) (arcsec)")
    if ylim_sdiq:
        ax_sdiq.set_ylim(*ylim_sdiq)
    ax_sdiq.set_xlim(xmin, xmax)
    ax_sdiq.legend(fontsize=9)
    ax_sdiq.tick_params(labelbottom=False)
    for m in metrics:
        xc = xcol[m.tag] if isinstance(xcol, dict) else xcol
        ok = df[m.sys2].notna() & df[xc].notna()
        if mask is not None:
            ok &= mask
        ax_cnt.hist(df.loc[ok, xc], bins=edges, histtype="step", lw=1.8, color=m.color)
    ax_cnt.set_xlabel(xlabel)
    ax_cnt.set_ylabel("visits")
    if title:
        fig.suptitle(title)
    return fig, tables, dict(hex=hex_axes, sdiq=ax_sdiq, counts=ax_cnt)


def binned_table(tables, metrics, column="sdiq"):
    """Side-by-side binned values (default sDIQ) for every metric, plus counts."""
    out = pd.DataFrame({"center": next(iter(tables.values()))["center"]}).set_index(
        "center"
    )
    for m in metrics:
        out[f"{column}_{m.tag}"] = tables[m.tag][column].to_numpy()
        out[f"n_{m.tag}"] = tables[m.tag]["n"].to_numpy()
    return out

## Parameters and data

In [ ]:
start_dayobs = parse_day_obs(start_dayobs)
end_dayobs = parse_day_obs(end_dayobs)
if start_dayobs > end_dayobs:
    raise ValueError(
        f"start_dayobs ({start_dayobs}) must be <= end_dayobs ({end_dayobs})"
    )
programs = parse_programs(science_programs)
dimm_outer_scale_m = float(dimm_outer_scale_m)
dimm_min_wind_speed = float(dimm_min_wind_speed)
ringss_window_minutes = float(ringss_window_minutes)
sdiq_goal_arcsec = (
    float(sdiq_goal_arcsec) if sdiq_goal_arcsec not in (None, "", "none") else None
)
use_cache = parse_bool(use_cache)

METRICS = make_metrics(dimm_outer_scale_m, dimm_min_wind_speed)
DIMM, RINGSS = METRICS

cache_tag = f"{CACHE_VERSION}_{start_dayobs}_{end_dayobs}_{'_'.join(p.replace('BLOCK-', '') for p in programs)}"
cache_dir = (
    os.path.join(NOTEBOOK_DIR, "cache") if (use_cache and not_times_square) else None
)

print(f"Nights {start_dayobs}-{end_dayobs}; programs {programs}")
print("Atmospheric metrics:", [m.label for m in METRICS])

In [ ]:
consdb = get_consdb_client()
raw = cached(
    cache_dir and os.path.join(cache_dir, f"visits_{cache_tag}.parquet"),
    lambda: fetch_visits(consdb, start_dayobs, end_dayobs, programs),
    use_cache,
)
ringss = await load_ringss(
    efd_name,
    start_dayobs,
    end_dayobs,
    cache_dir
    and os.path.join(cache_dir, f"ringss_{start_dayobs}_{end_dayobs}.parquet"),
    use_cache,
)
visits = match_ringss_to_visits(raw, ringss, ringss_window_minutes)

print(
    f"{len(visits)} visits on {visits['day_obs'].nunique()} nights; RINGSS source: "
    f"{visits['ringss_source'].iloc[0] if len(visits) else 'n/a'}"
)
visits.head()

In [ ]:
# What came back, per program and band, and how complete the key columns are.
if len(visits):
    print(pd.crosstab(visits["science_program"], visits["band"], margins=True))
print("\nfraction of visits with a value:")
print(
    visits[
        [
            "psf_sigma_median",
            "dimm_seeing",
            "ringss_scint",
            "wind_speed",
            "wind_dir",
            "aos_fwhm",
        ]
    ]
    .notna()
    .mean()
    .round(3)
)
if ringss is not None and len(ringss):
    print(
        f"\nRINGSS (EFD): {len(ringss)} measurements, {ringss['time'].min()} to {ringss['time'].max()}; "
        f"median cadence {ringss['time'].diff().median()}; visits with >= 1 match: {(visits['ringss_n'] > 0).mean():.2f}"
    )

## Derived quantities and quality cuts

Visits need a PSF measurement in a sane range, airmass ≤ 2.5 and a wind-speed reading; visits taken while
closing the AOS loop after a filter change are dropped. Each metric is then judged on its own valid sample:
the DIMM one needs a DIMM reading in range **and** wind ≥ `dimm_min_wind_speed`, the RINGSS one a RINGSS
reading in range. Set `require_both=True` to compare both metrics on exactly the same visits.

In [ ]:
visits = add_derived_columns(visits, METRICS, dimm_outer_scale_m, dimm_min_wind_speed)

cuts = QualityCuts(
    fwhm_range=(0.4, 4.0),
    dimm_range=(0.3, 4.0),
    ringss_range=(0.2, 4.0),
    airmass_max=2.5,
    exclude_bands=(),  # e.g. ("y",)
    exclude_reasons=("filter_change_close_loop",),
    require_both=False,
)
sel = apply_quality_cuts(visits, METRICS, cuts)

In [ ]:
# Sanity check on the de-correction: the factor applied to a zenith / 500 nm seeing for each band at a few airmasses.
probe = pd.DataFrame(
    {
        "airmass": np.repeat([1.0, 1.2, 1.5, 2.0], len(BAND_ORDER)),
        "band": list(BAND_ORDER) * 4,
    }
)
probe["factor"] = seeing_zenith_500nm_to_observed(1.0, probe["airmass"], probe["band"])
probe.pivot(index="airmass", columns="band", values="factor")[list(BAND_ORDER)].round(3)

## Headline numbers

sDIQ over the whole range for each atmospheric metric (`sdiq`, arcsec, with `sdiq_err`), alongside the
medians of the ingredients and the spread of the per-visit squared term. `frac_sys2_negative` is the fraction
of visits where the monitor seeing exceeds the delivered FWHM.

In [ ]:
headline = summary_table(sel, METRICS)
for m in METRICS:
    row = headline.loc[m.label]
    print(
        f'{m.label}: sDIQ = {row["sdiq"]:.3f} +/- {row["sdiq_err"]:.3f} arcsec  '
        f'(median sys2 {row["median_sys2"]:+.4f} arcsec2, n = {int(row["n_visits"])} visits on {int(row["n_nights"])} nights)'
    )
headline.round(4).T

In [ ]:
# The same per band (a band-dependent sDIQ would point at the lambda^0.2 de-correction or band-specific optics terms).
summary_by_group(sel, METRICS, "band").round(3)

## Histogram of the squared system term

Per-visit $\mathrm{FWHM}^2 - \mathrm{atm}^2$ for each metric. The dotted lines are the medians whose square
roots are the headline sDIQ values; the dashed green line is the goal, squared.

In [ ]:
fig = plot_sys2_histogram(
    sel, METRICS, bins=np.arange(-1.0, 2.0, 0.025), goal=sdiq_goal_arcsec
)
plt.title(f"{start_dayobs}-{end_dayobs}, {', '.join(programs)}")
plt.show()

## Nightly time series

Top: nightly median delivered FWHM and each monitor's seeing after de-correction to the visit airmass and
band. Middle: the nightly sDIQ per metric (square root of that night's median $\mathrm{sys}^2$; nights with
fewer than 5 valid visits for a metric are skipped). Bottom: visits per night and the median wind.

In [ ]:
fig, nightly = plot_nightly(sel, METRICS, goal=sdiq_goal_arcsec, min_count=5)
plt.show()

nightly_sdiq(sel, METRICS, min_count=5).set_index("day_obs").drop(columns="date").round(
    3
)

## Cross terms

Each figure below shows, per metric, the per-visit $\mathrm{sys}^2$ against one variable as a hexbin density
(log colour) with the binned median and 16–84 % band, then the binned sDIQ (square root of the binned median)
for both metrics on one axis, and the visit counts per bin. Bins with fewer than 10 visits are left empty.

In [ ]:
# sDIQ versus outside wind speed. The DIMM metric only exists above dimm_min_wind_speed (shaded).
wind_bins = np.arange(0, 20.1, 1.0)
fig, wind_tables, axes = plot_sdiq_vs(
    sel,
    METRICS,
    "wind_speed",
    wind_bins,
    "wind speed (m/s)",
    goal=sdiq_goal_arcsec,
    title="sDIQ vs wind speed",
)
for ax in axes["hex"] + [axes["sdiq"]]:
    ax.axvspan(wind_bins[0], dimm_min_wind_speed, color=INK_MUTED, alpha=0.08, lw=0)
plt.show()

binned_table(wind_tables, METRICS).round(3)

In [ ]:
# sDIQ versus pointing relative to the wind (telescope azimuth - wind direction; 0 = into the wind),
# restricted to windy visits (wind >= dimm_min_wind_speed) where the direction matters.
az_bins = np.arange(-180, 181, 30)
windy = sel["wind_speed"] >= dimm_min_wind_speed
fig, az_tables, axes = plot_sdiq_vs(
    sel,
    METRICS,
    "rel_wind_az",
    az_bins,
    "telescope azimuth - wind direction (deg; 0 = into the wind)",
    goal=sdiq_goal_arcsec,
    mask=windy,
    title=f"sDIQ vs pointing relative to the wind (wind >= {dimm_min_wind_speed:g} m/s)",
)
for ax in axes["hex"] + [axes["sdiq"], axes["counts"]]:
    ax.set_xticks(np.arange(-180, 181, 60))
plt.show()

binned_table(az_tables, METRICS).round(3)

In [ ]:
# sDIQ versus the AOS-estimated optics FWHM. If the system term were only the optics, sDIQ would follow the 1:1
# line (or the camera-diffusion-added line sqrt(aos_fwhm^2 + CAM_FWHM^2)); a constant excess points elsewhere
# (dome / mirror seeing, wind shake, tracking).
aos_bins = np.arange(0.0, 1.01, 0.05)
aos_grid = np.linspace(aos_bins[0], aos_bins[-1], 50)
fig, aos_tables, _ = plot_sdiq_vs(
    sel,
    METRICS,
    "aos_fwhm",
    aos_bins,
    "aos_fwhm (arcsec)",
    goal=sdiq_goal_arcsec,
    ref_lines=[
        (aos_grid, aos_grid, "sDIQ = aos_fwhm"),
        (
            aos_grid,
            np.sqrt(aos_grid**2 + CAM_FWHM**2),
            f"sqrt(aos_fwhm² + {CAM_FWHM}²)",
        ),
    ],
    title="sDIQ vs aos_fwhm",
)
plt.show()

binned_table(aos_tables, METRICS).round(3)

In [ ]:
# sDIQ versus the atmospheric seeing metric itself (each metric against its own de-corrected value).
# A trend here means the subtraction is not clean: a slope < 0 that the monitor over-predicts in bad seeing
# (or an outer-scale / wavelength scaling mismatch), a slope > 0 that part of the "atmosphere" is missed.
atm_bins = np.arange(0.3, 2.01, 0.1)
fig, atm_tables, _ = plot_sdiq_vs(
    sel,
    METRICS,
    {m.tag: m.atm for m in METRICS},
    atm_bins,
    "atmospheric seeing, de-corrected to visit airmass/band (arcsec)",
    goal=sdiq_goal_arcsec,
    title="sDIQ vs the atmospheric seeing metric",
)
plt.show()

binned_table(atm_tables, METRICS).round(3)

## Notes

* The two metrics are not judged on the same visits unless `require_both=True`: the DIMM one drops every
  visit below `dimm_min_wind_speed`, so its wind-speed curve starts there and its headline value is a
  windy-conditions number.
* Both monitors sit outside the dome, so $\mathrm{sys}^2$ bundles dome and mirror seeing, wind shake, tracking
  and optics (`aos_fwhm`) together with any monitor calibration error. A negative median means the monitor
  over-predicts the free-atmosphere contribution for that set of visits.
* The de-correction uses the same $X^{0.6}$ airmass and $(\lambda/500\,\mathrm{nm})^{-0.2}$ bandpass scaling as
  `lsst.summit.utils`; `seeing_zenith_500nm_median` from the ConsDB is not used because it has been empty
  for most of early operations.
* Changes to the method go through the
  [PTH Method Change Process](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/2032959501).